# Study WFS Mimic (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-06-04  
**Last Modified:** 2026-06-04  
**Status:** Draft  
**Keywords:** WFS, wavefront sensor, double Zernike, DOF recovery, FAM mimic  

## Description

Study whether the 4 corner wavefront sensors (WFS) alone can reconstruct
the optical state (degrees of freedom) that the full-array mode (FAM)
analysis provides.  Since actual WFS data is not available for the FAM
observations, we **mimic** WFS measurements by averaging FAM donut
Zernikes in annular wedge regions at the WFS field radius (~1.65 deg).

Key functionality:
1. Build per-visit "WFS mimic" Zernike vectors from FAM donut data
2. Subtract the measured intrinsic wavefront at the WFS positions
3. Build the OFC sensitivity matrix in the WFS-Zernike basis via SVD
4. Recover per-visit DOFs from the WFS mimic and compare against FAM DOFs

**Output:** Diagnostic plots, DOF comparison vs FAM analysis

**Based on:** `build_measured_intrinsic.ipynb`, `smatrix_doublez.ipynb`

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-04 | Aaron Roodman | Initial version |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Loading](#data)
5. [Z4 Height Correction](#z4height)
6. [WFS Mimic Construction](#wfs-mimic)
7. [Sensitivity Matrix & SVD](#svd)
8. [DOF Recovery from WFS Mimic](#dof-recovery)
9. [Comparison: WFS vs FAM DOFs](#comparison)
10. [PSF FWHM-Equivalent: Before vs After DOF Adjustment](#fwhm)

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# ----- Input from build_measured_intrinsic -----
input_dir = 'output/build_measured_intrinsic/bin_2x_nkeep_34_OCS_riz_alt_55.0_75.0_rot_-3.0_3.0'
path_tag = 'pathA'

# ----- Binning selector (for donut parquet resolution) -----
binning = 'bin_2x'

# ----- Coordinate system -----
coord_sys = 'OCS'

# ----- WFS mimic geometry -----
# Annular wedge: radial range and azimuthal half-width
wfs_inner_radius_deg = 1.60     # inner edge of annular wedge
wfs_outer_radius_deg = 1.725    # outer edge of annular wedge
wfs_azimuth_width_deg = 7.5     # full azimuthal extent of each wedge (+/- 3.75 deg)

# Azimuthal offset for the 4 WFS positions.
# The 4 WFS centers are placed at delta + [0, 90, 180, 270] degrees
# on the focal plane.  Settable from -40 to +40 degrees.
delta_deg = 0.0

# ----- OFC sensitivity-matrix / SVD parameters -----
# Same defaults as build_measured_intrinsic.
k_min, k_max = 1, 6
n_keep = 34

# (n_dof, n_keep) sets for the WFS-vs-FAM DOF comparison.  Each set
# restricts the OFC sensitivity matrix to an explicit subset of the 50
# degrees of freedom (DOF_INDEX_SETS, indexed into LABELS_50DOF) and
# keeps n_keep SVD modes for DOF recovery.  All of the WFS-vs-FAM
# comparison and FWHM plots are repeated for each set.
DOF_INDEX_SETS = {
    50: list(range(50)),                  # all DOFs
    22: (list(range(0, 10))               # all 10 hexapod (M2 + Cam rigid body)
         + list(range(10, 17))            # first 7 M1M3 bending modes
         + list(range(30, 35))),          # first 5 M2 bending modes
}
dof_keep_sets = [
    (50, 34),
    (22, 12),
]

# OFC config — auto-detected from RSP location.
ofc_config_subpath = 'ts_config_mttcs/MTAOS/v13/ofc'

# Path to the OFC normalization-weights yaml (50-DoF baseline).
# None -> auto-discover from $TS_CONFIG_MTTCS_DIR.
ofc_normalization_yaml = None

# Focal-plane Zernike normalization radius
fp_radius_basis = 1.75

# ----- PSF FWHM-equivalent (Section 10) -----
# The OFC sensitivity matrix is a smooth function of field position, so
# rather than calling dz_sens.evaluate at every donut (slow) we evaluate
# it once on a coarse focal-plane grid and bilinearly interpolate to each
# donut.  fwhm_field_grid_n = number of grid points across the field
# diameter (2 * fp_radius_basis); 15 -> ~0.25 deg spacing.
fwhm_field_grid_n = 15

# ----- Z4 CCD-height correction -----
height_map_fits = 'output/LSST_FP_cold_b_measurement_4col_bysurface.fits'
height_to_z4_factor = None    # None -> ccd_height.HEIGHT_TO_Z4_UM_PER_MM
intrinsic_transpose_bug = True

# ----- Output -----
output_dir = None

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from astropy.table import QTable, vstack as table_vstack
from scipy.interpolate import RegularGridInterpolator
import yaml

# Make this notebook runnable from the aos/ directory.
if not (Path.cwd() / 'code' / 'measured_intrinsic.py').exists():
    for _anc in Path.cwd().parents:
        if (_anc / 'code' / 'measured_intrinsic.py').exists():
            os.chdir(_anc)
            print(f'(chdir -> {_anc})')
            break

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting, detect_rsp_location, get_packages_dir
from common.zernike_names import NOLL_NAMES

setup_plotting()

sys.path.insert(0, 'code')
from dz_fitting import derive_noll_indices, focal_plane_zernike_basis
from measured_intrinsic import (
    apply_visit_filters,
    bin_median_focal,
    interpolate_grid_at_donuts,
)
from intrinsics_lib import (
    build_visit_marker_lookup, visit_marker_style,
)
from run_pipeline import (
    load_runs as _rp_load_runs,
    load_param_sets as _rp_load_param_sets,
    resolve_run as _rp_resolve_run,
    donut_path as _rp_donut_path,
)

# OFC / ts_ofc imports (RSP only)
from lsst.ts.ofc import OFCData, SensitivityMatrix

# CCD height-map helpers (RSP only)
from ccd_height import (
    compute_fp_coords,
    make_metrology_table,
    get_height_interpolator,
    height_to_z4,
    transpose_around_ccd_centers,
    HEIGHT_TO_Z4_UM_PER_MM,
)
from lsst.obs.lsst import LsstCam

print('All imports OK.')

# ---- Collect every figure into a single PDF in the output area ----
save_pdf = True
if output_dir is None:
    output_dir = 'output/study_wfs_mimic'
Path(output_dir).mkdir(parents=True, exist_ok=True)
pdf_path = str(Path(output_dir) / 'study_wfs_mimic_plots.pdf')
_pdf = PdfPages(pdf_path) if save_pdf else None


def _save_and_show(fig):
    """Save a figure to the collected PDF (if open) and display it."""
    if _pdf is not None:
        try:
            _pdf.savefig(fig, bbox_inches='tight')
        except Exception as _e:
            print(f'(pdf savefig failed: {_e})')
    plt.show()


if _pdf is not None:
    print(f'Collecting all plots into {pdf_path}')

<a id='functions'></a>
## 3. Helper Functions

### MeasuredIntrinsicGrid

A class that reads the measured-intrinsic grid parquet produced by
`build_measured_intrinsic` and provides bilinear interpolation of the
intrinsic Zernikes at arbitrary (thx_deg, thy_deg) positions in the
OCS coordinate system.  This is used to evaluate the intrinsic wavefront
at the WFS mimic positions.

In [ ]:
class MeasuredIntrinsicGrid:
    """Interpolator for the measured-intrinsic wavefront grid.

    Reads the grid parquet (from build_measured_intrinsic) and builds
    one RegularGridInterpolator per Zernike index.  Provides
    `evaluate(thx_deg, thy_deg)` returning shape (n_points, n_zk).
    """

    def __init__(self, grid_parquet_path):
        tbl = QTable.read(str(grid_parquet_path), format='parquet')
        self.coord_sys = str(tbl['coord_sys'][0])
        self.nollIndices = np.array(tbl['nollIndices'][0])
        self.iZs = [int(n) for n in self.nollIndices]
        self.iZidx = {iZ: i for i, iZ in enumerate(self.iZs)}
        self.n_zk = len(self.iZs)

        thx_all = np.array(tbl['thx_deg'], dtype=float)
        thy_all = np.array(tbl['thy_deg'], dtype=float)
        zk_all = np.array([np.array(row) for row in tbl['zk']])

        thx_unique = np.sort(np.unique(np.round(thx_all, 8)))
        thy_unique = np.sort(np.unique(np.round(thy_all, 8)))
        self.thx_grid = thx_unique
        self.thy_grid = thy_unique
        nx, ny = len(thx_unique), len(thy_unique)

        self._interps = []
        for zi in range(self.n_zk):
            grid_2d = np.full((nx, ny), np.nan)
            for row_idx in range(len(tbl)):
                xi = np.searchsorted(thx_unique, np.round(thx_all[row_idx], 8))
                yi = np.searchsorted(thy_unique, np.round(thy_all[row_idx], 8))
                grid_2d[xi, yi] = zk_all[row_idx, zi]

            nan_mask = np.isnan(grid_2d)
            if nan_mask.any():
                grid_2d[nan_mask] = 0.0

            interp = RegularGridInterpolator(
                (thx_unique, thy_unique), grid_2d,
                method='linear', bounds_error=False, fill_value=np.nan)
            self._interps.append(interp)

        print(f'MeasuredIntrinsicGrid: {nx}x{ny} grid, '
              f'{self.n_zk} Zernikes, coord_sys={self.coord_sys}')

    def evaluate(self, thx_deg, thy_deg):
        """Interpolate intrinsic Zernikes at given positions.

        Parameters
        ----------
        thx_deg, thy_deg : array-like
            Field angles in degrees.

        Returns
        -------
        zk : ndarray, shape (n_points, n_zk)
        """
        thx = np.atleast_1d(np.asarray(thx_deg, dtype=float))
        thy = np.atleast_1d(np.asarray(thy_deg, dtype=float))
        pts = np.column_stack((thx, thy))
        result = np.column_stack([interp(pts) for interp in self._interps])
        return result

### WFS Mimic Measurement

For each visit, select donuts in 4 annular wedge regions at the WFS
field radius.  The wedges are centered at azimuthal angles
`delta + [0, 90, 180, 270]` degrees, with a configurable radial range
and azimuthal width.  The median Zernike over each wedge is the
"WFS mimic" measurement for that position.

In [ ]:
def mimic_wfs_measurement(donut_df, visit_mask, coord_sys,
                          wfs_inner_radius_deg, wfs_outer_radius_deg,
                          wfs_azimuth_width_deg, delta_deg):
    """Build 4 WFS-mimic Zernike vectors from donuts in annular wedges.

    Parameters
    ----------
    donut_df : DataFrame
        Full donut table (all visits).
    visit_mask : ndarray of bool
        Mask selecting donuts for this visit.
    coord_sys : str
        'OCS' or 'CCS'.
    wfs_inner_radius_deg, wfs_outer_radius_deg : float
        Radial bounds of the annular wedge (degrees).
    wfs_azimuth_width_deg : float
        Full azimuthal extent of each wedge (degrees).
    delta_deg : float
        Azimuthal offset of the wedge centers.

    Returns
    -------
    dict with keys:
        'thx_deg', 'thy_deg' : ndarray (4,) — wedge center positions
        'zk_median' : ndarray (4, n_zk) — median Zernike per wedge
        'n_donuts' : ndarray (4,) — donut count per wedge
    """
    thx_col = f'thx_{coord_sys}'
    thy_col = f'thy_{coord_sys}'
    zk_col = f'zk_{coord_sys}'

    thx_rad = np.asarray(donut_df.loc[visit_mask, thx_col], dtype=float)
    thy_rad = np.asarray(donut_df.loc[visit_mask, thy_col], dtype=float)
    thx_deg = np.degrees(thx_rad)
    thy_deg = np.degrees(thy_rad)
    zk_arr = np.stack(donut_df.loc[visit_mask, zk_col].values).astype(float)

    r_deg = np.hypot(thx_deg, thy_deg)
    az_deg = np.degrees(np.arctan2(thy_deg, thx_deg)) % 360.0

    wfs_centers_az = [(delta_deg + offset) % 360.0 for offset in [0, 90, 180, 270]]
    half_az = wfs_azimuth_width_deg / 2.0

    n_zk = zk_arr.shape[1]
    zk_median = np.full((4, n_zk), np.nan)
    thx_out = np.zeros(4)
    thy_out = np.zeros(4)
    n_donuts = np.zeros(4, dtype=int)

    radial_mask = (r_deg >= wfs_inner_radius_deg) & (r_deg <= wfs_outer_radius_deg)

    for i, az_center in enumerate(wfs_centers_az):
        az_lo = (az_center - half_az) % 360.0
        az_hi = (az_center + half_az) % 360.0
        if az_lo < az_hi:
            az_mask = (az_deg >= az_lo) & (az_deg <= az_hi)
        else:
            az_mask = (az_deg >= az_lo) | (az_deg <= az_hi)

        wedge_mask = radial_mask & az_mask
        n_donuts[i] = int(wedge_mask.sum())

        r_mid = 0.5 * (wfs_inner_radius_deg + wfs_outer_radius_deg)
        az_rad = np.radians(az_center)
        thx_out[i] = r_mid * np.cos(az_rad)
        thy_out[i] = r_mid * np.sin(az_rad)

        if n_donuts[i] >= 3:
            zk_median[i] = np.nanmedian(zk_arr[wedge_mask], axis=0)

    return {
        'thx_deg': thx_out,
        'thy_deg': thy_out,
        'zk_median': zk_median,
        'n_donuts': n_donuts,
    }

### Z4 Height Correction for WFS Mimic

The Z4 (defocus) Zernike has a contribution from the physical CCD height
variation across the focal plane.  For donuts that have different intra-
and extra-focal positions, we compute the Z4 height at both positions
and average them.  This is the correct treatment because the donut pair
measurement is sensitive to the average defocus at the two positions.

In [ ]:
def compute_z4_height_correction(donut_df, visit_mask, interp, camera,
                                  factor=HEIGHT_TO_Z4_UM_PER_MM,
                                  intrinsic_transpose_bug=True):
    """Compute per-donut Z4 height correction averaging intra+extra positions.

    Returns
    -------
    z4hgt : ndarray (n_donuts_in_mask,)
        Z4 height contribution (μm) — average of intra and extra.
    z4hgt_intrinsic : ndarray (n_donuts_in_mask,)
        Z4 height as used by the pipeline intrinsic (with or without
        the per-CCD transpose bug).
    """
    sub = donut_df.loc[visit_mask]

    # Focal-plane coords for intra and extra separately
    fpx_intra, fpy_intra = compute_fp_coords(
        sub, camera, x_col='centroid_x_intra', y_col='centroid_y_intra')
    fpx_extra, fpy_extra = compute_fp_coords(
        sub, camera, x_col='centroid_x_extra', y_col='centroid_y_extra')

    z4_intra = height_to_z4(interp(fpx_intra, fpy_intra), factor=factor)
    z4_extra = height_to_z4(interp(fpx_extra, fpy_extra), factor=factor)
    z4hgt = 0.5 * (z4_intra + z4_extra)

    if intrinsic_transpose_bug:
        det_names = np.asarray(sub['detector']).astype(str)
        fpx_t_intra, fpy_t_intra = transpose_around_ccd_centers(
            fpx_intra, fpy_intra, det_names, camera)
        fpx_t_extra, fpy_t_extra = transpose_around_ccd_centers(
            fpx_extra, fpy_extra, det_names, camera)
        z4_t_intra = height_to_z4(interp(fpx_t_intra, fpy_t_intra), factor=factor)
        z4_t_extra = height_to_z4(interp(fpx_t_extra, fpy_t_extra), factor=factor)
        z4hgt_intrinsic = 0.5 * (z4_t_intra + z4_t_extra)
    else:
        z4hgt_intrinsic = z4hgt

    return z4hgt, z4hgt_intrinsic

### DOF Recovery

Given U-mode amplitudes from the WFS SVD, recover physical degrees of
freedom using the V matrix and singular values, following the same
approach as `build_measured_intrinsic`.

In [ ]:
LABELS_50DOF = [
    'M2_dz', 'M2_dx', 'M2_dy', 'M2_rx', 'M2_ry',
    'Cam_dz', 'Cam_dx', 'Cam_dy', 'Cam_rx', 'Cam_ry',
    'B1_1', 'B1_2', 'B1_3', 'B1_4', 'B1_5',
    'B1_6', 'B1_7', 'B1_8', 'B1_9', 'B1_10',
    'B1_11', 'B1_12', 'B1_13', 'B1_14', 'B1_15',
    'B1_16', 'B1_17', 'B1_18', 'B1_19', 'B1_20',
    'B2_1', 'B2_2', 'B2_3', 'B2_4', 'B2_5',
    'B2_6', 'B2_7', 'B2_8', 'B2_9', 'B2_10',
    'B2_11', 'B2_12', 'B2_13', 'B2_14', 'B2_15',
    'B2_16', 'B2_17', 'B2_18', 'B2_19', 'B2_20',
]
DOF_UNITS_50 = (
    ['μm', 'μm', 'μm', 'arcsec', 'arcsec'] +
    ['μm', 'μm', 'μm', 'arcsec', 'arcsec'] +
    ['μm'] * 20 +
    ['μm'] * 20
)


def recover_dof_per_visit(A_modes, V, Sigma, N_diag, n_keep):
    """Recover physical DOF estimates per visit from U-mode amplitudes.

    Parameters
    ----------
    A_modes : ndarray (n_visits, n_keep)
        U-mode amplitudes per visit (= U_eff^T @ w).
    V : ndarray (n_dof, n_dof)
        Right singular vectors of S.
    Sigma : ndarray (n_dof,)
        Singular values.
    N_diag : ndarray (n_dof,)
        OFC normalization weights (diagonal of N).
    n_keep : int

    Returns
    -------
    dof : ndarray (n_visits, n_dof)
        Physical DOF (mixed units per DOF_UNITS_50).
    """
    V_eff = V[:, :n_keep]
    inv_sig = 1.0 / Sigma[:n_keep]
    A = np.where(np.isfinite(A_modes), A_modes, 0.0)
    dof_norm = (V_eff * inv_sig[None, :]) @ A.T
    dof_phys = N_diag[:, None] * dof_norm
    return dof_phys.T

### Donut Parquet Loading

Resolve and load the multi-chunk donut parquet files, using the same
infrastructure as `build_measured_intrinsic`.

In [ ]:
def resolve_chunk_parquets(binning, donut_parquets=None):
    """Return the list of donut parquet paths for the given binning."""
    if donut_parquets:
        return [Path(p) for p in donut_parquets]
    target_ps = f'fam_danish_v1_triplets_{binning}'
    runs = _rp_load_runs().get('runs', {})
    param_sets = _rp_load_param_sets()
    paths = []
    for name, cfg in runs.items():
        if cfg.get('param_set') != target_ps:
            continue
        resolved = _rp_resolve_run(cfg, param_sets)
        p = Path(_rp_donut_path(resolved))
        if p.exists():
            paths.append(p)
        else:
            print(f'  (skipping {name}: {p} not found)')
    if not paths:
        raise FileNotFoundError(
            f'No donut parquet files resolved for binning={binning!r}')
    return paths


def visits_sidecar_path(donut_parquet_path):
    p = Path(donut_parquet_path)
    return p.with_name(p.stem + '_visits.parquet')


def load_chunks(donut_parquet_paths):
    """Read donut + visits sidecars across chunks."""
    donut_frames = []
    visits_tables = []
    for p in donut_parquet_paths:
        vp = visits_sidecar_path(p)
        assert p.exists(), f'missing {p}'
        assert vp.exists(), f'missing {vp}'
        d = pd.read_parquet(p)
        v = QTable.read(str(vp), format='parquet')
        donut_frames.append(d)
        visits_tables.append(v)
        print(f'  {p.name}:  {len(d):>8d} donuts,  {len(v):>4d} visits')
    donut_df = pd.concat(donut_frames, ignore_index=True)
    visits_full = (visits_tables[0]
                   if len(visits_tables) == 1
                   else table_vstack(visits_tables, join_type='exact'))
    return donut_df, visits_full


def filter_donuts_by_visits(donut_df, visits_kept):
    """Return rows of donut_df whose (day_obs, seq_num) appears in visits_kept."""
    keep_keys = set(zip(
        np.asarray(visits_kept['day_obs']).tolist(),
        np.asarray(visits_kept['seq_num']).tolist(),
    ))
    keys = list(zip(donut_df['day_obs'].tolist(),
                    donut_df['seq_num'].tolist()))
    mask = np.array([k in keep_keys for k in keys])
    return donut_df.loc[mask].reset_index(drop=True)

<a id='data'></a>
## 4. Data Loading

Load three data sources:
1. **DZ fits parquet** from `build_measured_intrinsic` — per-visit FAM
   analysis results including DZ coefficients, u/v-mode amplitudes, and
   recovered DOFs.  These are the "truth" for comparison.
2. **Measured intrinsic grid parquet** — the focal-plane wavefront
   intrinsic on a regular grid, used to subtract the intrinsic from the
   WFS mimic measurements.
3. **Donut parquet files** — the raw per-donut Zernike measurements from
   FAM, which we average in WFS-like regions.

In [ ]:
# --- Load FAM analysis results ---
input_path = Path(input_dir)
dz_fits_path = sorted(input_path.glob(f'*_{path_tag}_dz_fits.parquet'))
grid_path = sorted(input_path.glob(f'*_{path_tag}_grid.parquet'))

assert len(dz_fits_path) == 1, f'Expected 1 dz_fits file, found {len(dz_fits_path)}'
assert len(grid_path) == 1, f'Expected 1 grid file, found {len(grid_path)}'

dz_df = pd.read_parquet(dz_fits_path[0])
print(f'FAM dz_fits: {len(dz_df)} visits, {len(dz_df.columns)} columns')
print(f'  day_obs range: {dz_df["day_obs"].min()} - {dz_df["day_obs"].max()}')

intrinsic_grid = MeasuredIntrinsicGrid(grid_path[0])

In [ ]:
# --- Load donut parquets ---
donut_parquet_paths = resolve_chunk_parquets(binning)
print(f'Using binning = {binning!r};  {len(donut_parquet_paths)} chunk(s):')
for p in donut_parquet_paths:
    print(f'   {p}')

donut_full, visits_full = load_chunks(donut_parquet_paths)
print(f'\nTotal: {len(donut_full)} donuts, {len(visits_full)} visits')

# Filter to visits that are in the FAM analysis
fam_keys = set(zip(dz_df['day_obs'].tolist(), dz_df['seq_num'].tolist()))
visits_kept_mask = np.array([
    (int(d), int(s)) in fam_keys
    for d, s in zip(np.asarray(visits_full['day_obs']),
                    np.asarray(visits_full['seq_num']))
])
visits_kept = visits_full[visits_kept_mask]
print(f'Visits matching FAM analysis: {len(visits_kept)}/{len(visits_full)}')

donut_df = filter_donuts_by_visits(donut_full, visits_kept)
print(f'Donuts after filter: {len(donut_df)}/{len(donut_full)}')

# Derive Noll indices
nZk = np.stack(donut_df[f'zk_{coord_sys}'].values).shape[1]
noll_arr = (np.array(visits_kept['nollIndices'][0])
            if 'nollIndices' in visits_kept.colnames else None)
iZs, iZidx = derive_noll_indices(nZk, noll_arr)
iZs_arr = np.array(iZs, dtype=int)
n_j = len(iZs_arr)
print(f'Pupil Noll indices ({n_j}): {iZs_arr}')

<a id='z4height'></a>
## 5. Z4 Height Correction

The Z4 (defocus) Zernike includes a contribution from the physical
height variation of CCDs across the focal plane.  We precompute the Z4
height correction for every donut, averaging the height at the intra-
and extra-focal donut positions.  This correction is applied to the
*measured* Zernike before comparing against the intrinsic (which already
has the height contribution baked in, including any pipeline bugs).

In [ ]:
camera = LsstCam.getCamera()

z4_can_correct = (height_map_fits
                  and Path(height_map_fits).exists()
                  and 4 in iZs)

Z4hgt = None
Z4hgt_intrinsic = None

if z4_can_correct:
    print(f'Loading metrology: {height_map_fits}')
    metrology = make_metrology_table(height_map_fits)
    print(f'  {len(metrology)} metrology points across '
          f'{len(set(metrology["det"]))} sensors')

    factor = (height_to_z4_factor
              if height_to_z4_factor is not None
              else HEIGHT_TO_Z4_UM_PER_MM)
    interp_height = get_height_interpolator(metrology)

    # Per-donut Z4 height correction, averaging intra + extra positions
    Z4hgt, Z4hgt_intrinsic = compute_z4_height_correction(
        donut_df, np.ones(len(donut_df), dtype=bool),
        interp_height, camera,
        factor=factor,
        intrinsic_transpose_bug=intrinsic_transpose_bug)

    print(f'Z4hgt (intra+extra avg): '
          f'mean={np.nanmean(Z4hgt):+.4f} μm, '
          f'std={np.nanstd(Z4hgt):.4f} μm')
    print(f'Z4hgt_intrinsic:         '
          f'mean={np.nanmean(Z4hgt_intrinsic):+.4f} μm, '
          f'std={np.nanstd(Z4hgt_intrinsic):.4f} μm')
else:
    print('Skipping Z4 height correction')

<a id='wfs-mimic'></a>
## 6. WFS Mimic Construction

For each visit, we:
1. Select donuts in 4 annular wedge regions at the WFS field radius
2. Compute the median Zernike across each wedge
3. Apply Z4 height correction if available
4. Subtract the measured intrinsic at the wedge center positions
5. The result is a "WFS deviation" vector per visit — the signal that
   the AOS control loop would act on

The WFS deviation captures both the correctable optical state (DOFs)
and any uncorrectable residual.  A key question is how well the DOFs
can be recovered from just 4 field positions.

In [ ]:
# Build per-visit WFS mimic measurements
dobs_arr = np.asarray(donut_df['day_obs'])
snum_arr = np.asarray(donut_df['seq_num'])

# Sort dz_df to ensure consistent ordering
dz_df = dz_df.sort_values(['day_obs', 'seq_num']).reset_index(drop=True)
n_visits = len(dz_df)

wfs_results = []
for v_idx in range(n_visits):
    d = int(dz_df.iloc[v_idx]['day_obs'])
    s = int(dz_df.iloc[v_idx]['seq_num'])
    visit_mask = (dobs_arr == d) & (snum_arr == s)

    result = mimic_wfs_measurement(
        donut_df, visit_mask, coord_sys,
        wfs_inner_radius_deg, wfs_outer_radius_deg,
        wfs_azimuth_width_deg, delta_deg)
    result['day_obs'] = d
    result['seq_num'] = s
    wfs_results.append(result)

    if v_idx == 0:
        print(f'Visit {d}/{s}: donuts per wedge = {result["n_donuts"]}')

# Summary
all_counts = np.array([r['n_donuts'] for r in wfs_results])
print(f'\n{n_visits} visits processed.')
print(f'Donuts per wedge — min: {all_counts.min(axis=0)}, '
      f'median: {np.median(all_counts, axis=0).astype(int)}, '
      f'max: {all_counts.max(axis=0)}')
low_count = np.any(all_counts < 3, axis=1)
print(f'Visits with <3 donuts in any wedge: {low_count.sum()}/{n_visits}')

### WFS Mimic Visualization

Show the donut distribution and WFS wedge regions for an example visit.

In [ ]:
from matplotlib.patches import Polygon


def annular_wedge_verts(az_center_deg, half_az_deg, r_in, r_out, n=60):
    """Vertices (in plot coords x=thy, y=thx) of an annular wedge outline.

    Azimuth is defined as in the mimic selection: az = atan2(thy, thx),
    so a point at azimuth ``a`` and radius ``r`` plots at
    (x=thy=r*sin(a), y=thx=r*cos(a)).
    """
    az = np.radians(np.linspace(az_center_deg - half_az_deg,
                                az_center_deg + half_az_deg, n))
    x_in, y_in = r_in * np.sin(az), r_in * np.cos(az)
    x_out, y_out = r_out * np.sin(az), r_out * np.cos(az)
    xs = np.concatenate([x_in, x_out[::-1]])
    ys = np.concatenate([y_in, y_out[::-1]])
    return np.column_stack([xs, ys])


# Pick an example visit where all 4 wedges are populated (>=3 donuts),
# so the validation plot shows a representative, fully-sampled case.
all_counts = np.array([r['n_donuts'] for r in wfs_results])
_good_visits = np.where(np.all(all_counts >= 3, axis=1))[0]
ex_idx = int(_good_visits[0]) if len(_good_visits) else 0

ex = wfs_results[ex_idx]
d, s = ex['day_obs'], ex['seq_num']
visit_mask = (dobs_arr == d) & (snum_arr == s)
thx_v = np.degrees(np.asarray(donut_df.loc[visit_mask, f'thx_{coord_sys}'], dtype=float))
thy_v = np.degrees(np.asarray(donut_df.loc[visit_mask, f'thy_{coord_sys}'], dtype=float))

fig, ax = plt.subplots(figsize=(8, 8), layout='constrained')
# Donut locations — larger, darker points
ax.plot(thy_v, thx_v, '.', ms=3, alpha=0.6, color='0.3', label='donuts')

colors = ['tab:red', 'tab:blue', 'tab:green', 'tab:orange']
half_az = wfs_azimuth_width_deg / 2.0
wfs_centers_az = [(delta_deg + offset) % 360.0 for offset in [0, 90, 180, 270]]

for i in range(4):
    # Open wedge showing the exact (radial x azimuthal) selection region
    verts = annular_wedge_verts(wfs_centers_az[i], half_az,
                                wfs_inner_radius_deg, wfs_outer_radius_deg)
    ax.add_patch(Polygon(verts, closed=True, fill=False,
                         edgecolor=colors[i], lw=2.0,
                         label=f'WFS {i} (n={ex["n_donuts"][i]})'))

ax.set_xlabel(f'thy_{coord_sys} [deg]')
ax.set_ylabel(f'thx_{coord_sys} [deg]')
ax.set_title(f'WFS Mimic Regions — visit {d}/{s}, delta={delta_deg:.0f}°')
ax.set_aspect('equal')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
_save_and_show(fig)

### Compute WFS Deviation Vectors

Subtract the measured intrinsic wavefront (from the grid) at the WFS
positions to get the deviation — this is the "measurement" that the
OFC would use to correct the optical state.

In [ ]:
# Compute deviation = measured_zk - intrinsic(at WFS positions)
# Apply Z4 height correction to measured Zernikes first if available

n_wfs_pts = 4
wfs_deviation = np.full((n_visits, n_wfs_pts, n_j), np.nan)
wfs_measured = np.full((n_visits, n_wfs_pts, n_j), np.nan)
wfs_intrinsic = np.full((n_visits, n_wfs_pts, n_j), np.nan)

for v_idx in range(n_visits):
    r = wfs_results[v_idx]
    zk_meas = r['zk_median'].copy()  # (4, n_zk)

    # Apply Z4 height correction to measured Zernikes if available
    if Z4hgt is not None and 4 in iZidx:
        d, s = r['day_obs'], r['seq_num']
        visit_mask = (dobs_arr == d) & (snum_arr == s)

        # For Z4 correction of the WFS median: we need the median Z4hgt
        # in each wedge, matching how we compute the median Zernikes
        thx_donut = np.degrees(np.asarray(
            donut_df.loc[visit_mask, f'thx_{coord_sys}'], dtype=float))
        thy_donut = np.degrees(np.asarray(
            donut_df.loc[visit_mask, f'thy_{coord_sys}'], dtype=float))
        r_donut = np.hypot(thx_donut, thy_donut)
        az_donut = np.degrees(np.arctan2(thy_donut, thx_donut)) % 360.0
        z4hgt_visit = Z4hgt[visit_mask]
        z4hgt_intr_visit = Z4hgt_intrinsic[visit_mask]

        radial_mask = ((r_donut >= wfs_inner_radius_deg)
                       & (r_donut <= wfs_outer_radius_deg))
        half_az = wfs_azimuth_width_deg / 2.0
        wfs_centers_az = [(delta_deg + offset) % 360.0
                          for offset in [0, 90, 180, 270]]

        for i, az_center in enumerate(wfs_centers_az):
            az_lo = (az_center - half_az) % 360.0
            az_hi = (az_center + half_az) % 360.0
            if az_lo < az_hi:
                az_mask = (az_donut >= az_lo) & (az_donut <= az_hi)
            else:
                az_mask = (az_donut >= az_lo) | (az_donut <= az_hi)
            wedge = radial_mask & az_mask
            if wedge.sum() >= 3:
                # Correct measured Z4: subtract data Z4hgt
                z4_corr = np.nanmedian(z4hgt_visit[wedge])
                zk_meas[i, iZidx[4]] -= z4_corr

    # Intrinsic at WFS positions
    zk_intr = intrinsic_grid.evaluate(r['thx_deg'], r['thy_deg'])

    wfs_measured[v_idx] = zk_meas
    wfs_intrinsic[v_idx] = zk_intr
    wfs_deviation[v_idx] = zk_meas - zk_intr

print(f'WFS deviation array: {wfs_deviation.shape}  (n_visits, 4, n_zk)')
print(f'Deviation RMS per Zernike (across visits and WFS positions):')
for ji, j in enumerate(iZs):
    rms = np.sqrt(np.nanmean(wfs_deviation[:, :, ji]**2))
    print(f'  Z{j}: {rms:.4f} μm')

### WFS Deviation Example

Show measured, intrinsic, and deviation for one example visit at each
WFS position.

In [ ]:
# Bar chart: measured, intrinsic, deviation for example visit.
# Reuse ex_idx from the region plot (a visit with all 4 wedges populated),
# so every WFS panel — including WFS 0 — has measured/deviation bars and
# not just the intrinsic (which is always defined from the grid).
fig, axes = plt.subplots(2, 2, figsize=(14, 10), layout='constrained')
axes = axes.ravel()
j_labels = [f'Z{j}' for j in iZs]
x = np.arange(n_j)

for i in range(4):
    ax = axes[i]
    ax.bar(x - 0.25, wfs_measured[ex_idx, i], width=0.25,
           label='measured', color='tab:blue', alpha=0.8)
    ax.bar(x, wfs_intrinsic[ex_idx, i], width=0.25,
           label='intrinsic', color='tab:orange', alpha=0.8)
    ax.bar(x + 0.25, wfs_deviation[ex_idx, i], width=0.25,
           label='deviation', color='tab:green', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(j_labels, fontsize=7, rotation=45)
    ax.set_ylabel('μm')
    r = wfs_results[ex_idx]
    ax.set_title(f'WFS {i} — ({r["thx_deg"][i]:.2f}, {r["thy_deg"][i]:.2f})°, '
                 f'n={r["n_donuts"][i]}', fontsize=9)
    ax.axhline(0, color='k', lw=0.5)
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend(fontsize=8)

d, s = wfs_results[ex_idx]['day_obs'], wfs_results[ex_idx]['seq_num']
fig.suptitle(f'WFS Mimic — visit {d}/{s}, delta={delta_deg:.0f}°', fontsize=12)
_save_and_show(fig)

<a id='svd'></a>
## 7. Sensitivity Matrix & SVD

We build two representations of the OFC sensitivity matrix:

1. **WFS-Zernike basis**: The sensitivity matrix evaluated directly at
   the 4 WFS field positions, giving a `(4 × n_zk, n_dof)` matrix.
   This is the natural basis for the WFS measurement vector.

2. **Double Zernike (DZ) basis**: The standard `(n_k × n_j, n_dof)`
   sensitivity matrix used by `build_measured_intrinsic`.

Both are right-multiplied by the OFC normalization matrix and SVD'd.
The key comparison is how the singular value spectrum and reachability
differ between the two representations — the WFS basis has fewer
measurements (84 vs 126) and loses information about the focal-plane
Zernike structure beyond what 4 points can resolve.

In [ ]:
# Auto-detect RSP location for OFC config
rsp_location = detect_rsp_location()
ofc_config_dir = get_packages_dir(rsp_location) + '/' + ofc_config_subpath
print(f'OFC config dir: {ofc_config_dir}')

ofc_data = OFCData('lsst', config_dir=ofc_config_dir)

# --- OFC Normalization weights ---
def _find_ofc_normalization_yaml(user_path=None,
                                 name='range0.5_fwhm-0.15.yaml'):
    if user_path:
        p = Path(user_path)
        if not p.exists():
            raise FileNotFoundError(f'{p} does not exist')
        return p
    root = os.environ.get('TS_CONFIG_MTTCS_DIR')
    if not root:
        raise RuntimeError(
            'TS_CONFIG_MTTCS_DIR is not set. Run setup ts_config_mttcs first.')
    target = Path(root) / 'MTAOS' / 'ofc' / 'normalization_weights' / name
    if not target.exists():
        raise FileNotFoundError(f'{target} not found')
    return target

norm_yaml_path = _find_ofc_normalization_yaml(ofc_normalization_yaml)
print(f'OFC normalization yaml: {norm_yaml_path}')
with open(norm_yaml_path) as _fp:
    normalization_weights = np.array(yaml.safe_load(_fp))
N_diag = normalization_weights
normalization_matrix = np.diag(normalization_weights)
print(f'Normalization: {len(normalization_weights)} weights '
      f'(min={normalization_weights.min():.3g}, max={normalization_weights.max():.3g})')

### WFS Sensitivity Matrix

Evaluate the OFC sensitivity matrix at the 4 WFS mimic field positions
using `SensitivityMatrix.evaluate()`.  This gives the wavefront
response at each WFS position for each DOF perturbation.

In [ ]:
# WFS field positions (thx, thy in degrees)
wfs_thx = wfs_results[0]['thx_deg']
wfs_thy = wfs_results[0]['thy_deg']
wfs_field_angles = [[float(wfs_thx[i]), float(wfs_thy[i])] for i in range(4)]
print('WFS field positions (deg):')
for i, (tx, ty) in enumerate(wfs_field_angles):
    r = np.hypot(tx, ty)
    az = np.degrees(np.arctan2(ty, tx))
    print(f'  WFS {i}: ({tx:+.4f}, {ty:+.4f}), r={r:.4f}°, az={az:.1f}°')

# Evaluate sensitivity matrix at WFS positions (full DOF set)
dz_sens = SensitivityMatrix(ofc_data)
sens_3d = dz_sens.evaluate(wfs_field_angles, 0.0)
print(f'\nRaw sensitivity shape: {sens_3d.shape}  (n_wfs, n_zk_full, n_dof_full)')

# Slice to our Zernike selection
zn_idx = iZs_arr - 4
sens_wfs_3d = sens_3d[:, zn_idx, :]            # (4, n_j, n_dof_full)
n_dof_full = sens_wfs_3d.shape[2]
print(f'After Zernike selection: {sens_wfs_3d.shape}  (4, {n_j}, {n_dof_full})')


def resolve_dof_idx(n_dof_label):
    """Map a DOF-set label to the array of absolute DOF indices."""
    idx = DOF_INDEX_SETS.get(n_dof_label, list(range(n_dof_label)))
    return np.array([d for d in idx if d < n_dof_full], dtype=int)


def build_wfs_svd(n_dof_label, n_keep_use):
    """SVD of the WFS sensitivity matrix restricted to a DOF index subset."""
    dof_idx = resolve_dof_idx(n_dof_label)
    S = sens_wfs_3d[:, :, dof_idx].reshape(-1, len(dof_idx))
    N_diag = np.asarray(normalization_weights)[dof_idx]
    S = S @ np.diag(N_diag)
    U, Sig, Vh = np.linalg.svd(S, full_matrices=False)
    nk = int(min(n_keep_use, U.shape[1]))
    return dict(S=S, U=U, Sigma=Sig, V=Vh.T, U_eff=U[:, :nk],
                N_diag=N_diag, dof_idx=dof_idx, n_dof=len(dof_idx), n_keep=nk)

### DZ Sensitivity Matrix (reference)

Build the standard Double Zernike sensitivity matrix using the same
procedure as `build_measured_intrinsic`, for comparison.

In [ ]:
# DZ sensitivity matrix — from ofc_data.sensitivity_matrix
S_full = np.asarray(ofc_data.sensitivity_matrix)
print(f'S_full shape (field, pupil-j, DoF) = {S_full.shape}')
n_k = k_max - k_min + 1
kj_grid = [(int(k_min + ki), int(iZs_arr[ji]))
           for ki in range(n_k) for ji in range(n_j)]


def build_dz_svd(dof_idx, n_keep_use):
    """SVD of the Double-Zernike sensitivity matrix restricted to dof_idx."""
    dof_idx = np.asarray(dof_idx, dtype=int)
    S_slab = S_full[k_min:k_max + 1, iZs_arr, :][:, :, dof_idx]   # (n_k, n_j, ndof)
    N_diag = np.asarray(normalization_weights)[dof_idx]
    S = S_slab.reshape(-1, len(dof_idx)) @ np.diag(N_diag)
    U, Sig, Vh = np.linalg.svd(S, full_matrices=False)
    nk = int(min(n_keep_use, U.shape[1]))
    return dict(S=S, U=U, Sigma=Sig, V=Vh.T, U_eff=U[:, :nk],
                dof_idx=dof_idx, n_dof=len(dof_idx), n_keep=nk)


# Build WFS + DZ SVDs for every (n_dof, n_keep) set.  Reused by both the
# SVD diagnostic plots (Section 7) and DOF recovery (Section 8).
svd_sets = []
for n_dof_label, n_keep_use in dof_keep_sets:
    wfs = build_wfs_svd(n_dof_label, n_keep_use)
    dz = build_dz_svd(wfs['dof_idx'], n_keep_use)
    svd_sets.append(dict(n_dof=wfs['n_dof'], n_keep=wfs['n_keep'],
                         dof_idx=wfs['dof_idx'], wfs=wfs, dz=dz))
    print(f"set n_dof={wfs['n_dof']:>2d}, n_keep={wfs['n_keep']:>2d}: "
          f"WFS S{tuple(wfs['S'].shape)}, DZ S{tuple(dz['S'].shape)}, "
          f"Σ_wfs=[{wfs['Sigma'][-1]:.3g}, {wfs['Sigma'][0]:.3g}]")

### SVD Diagnostic Plots

Compare the singular value spectra, U-modes, and V matrices for the WFS
and DZ representations.  These are produced **once per `(n_dof, n_keep)`
set** in `svd_sets`, so the WFS-vs-DZ comparison can be inspected for
each DOF basis.

In [ ]:
# ---- SVD diagnostic plotting helpers (per DOF set) ----
def plot_singular_values(wfs, dz, label):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), layout='constrained')
    ax = axes[0]
    ax.semilogy(np.arange(1, len(wfs['Sigma']) + 1), wfs['Sigma'], 'o-', ms=4,
                label=f"WFS (4 pos, {wfs['S'].shape[0]} rows)")
    ax.semilogy(np.arange(1, len(dz['Sigma']) + 1), dz['Sigma'], 's-', ms=4,
                label=f"DZ (k={k_min}..{k_max}, {dz['S'].shape[0]} rows)")
    ax.axvline(wfs['n_keep'], color='gray', ls='--', alpha=0.5,
               label=f"n_keep={wfs['n_keep']}")
    ax.set_xlabel('Mode index')
    ax.set_ylabel('Singular value')
    ax.set_title('Singular Value Spectrum')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

    ax = axes[1]
    m = min(len(wfs['Sigma']), len(dz['Sigma']))
    ratio = wfs['Sigma'][:m] / dz['Sigma'][:m]
    ax.plot(np.arange(1, m + 1), ratio, 'o-', ms=4, color='tab:green')
    ax.set_xlabel('Mode index')
    ax.set_ylabel('σ_WFS / σ_DZ')
    ax.set_title('Singular Value Ratio (WFS / DZ)')
    ax.axhline(1, color='k', lw=0.5)
    ax.grid(alpha=0.3)
    fig.suptitle(f'Sensitivity Matrix SVD — WFS vs DZ  [{label}]', fontsize=12)
    _save_and_show(fig)


def plot_umodes(wfs, dz, label):
    n_show = min(wfs['n_keep'], 34)
    fig, axes = plt.subplots(1, 2, figsize=(16, 8), layout='constrained')

    ax = axes[0]
    im = ax.imshow(wfs['U_eff'][:, :n_show], aspect='auto', cmap='RdBu_r',
                   vmin=-0.3, vmax=0.3)
    ax.set_xlabel('v-mode index')
    ax.set_ylabel('WFS Zernike row')
    ax.set_title(f"U_wfs  ({wfs['S'].shape[0]} rows × {n_show} modes)")
    ax.set_yticks(np.arange(0, 4 * n_j, n_j))
    ax.set_yticklabels([f'WFS{s}' for s in range(4)], fontsize=9)
    for s in range(1, 4):
        ax.axhline(s * n_j - 0.5, color='k', lw=0.8)
    plt.colorbar(im, ax=ax, shrink=0.8)

    ax = axes[1]
    n_show_dz = min(dz['n_keep'], 34)
    im = ax.imshow(dz['U_eff'][:, :n_show_dz], aspect='auto', cmap='RdBu_r',
                   vmin=-0.3, vmax=0.3)
    ax.set_xlabel('v-mode index')
    ax.set_ylabel('DZ (k,j) row')
    ax.set_title(f"U_dz  ({dz['S'].shape[0]} rows × {n_show_dz} modes)")
    ax.set_yticks(np.arange(0, n_k * n_j, n_j))
    ax.set_yticklabels([f'k={k_min+ki}' for ki in range(n_k)], fontsize=9)
    for ki in range(1, n_k):
        ax.axhline(ki * n_j - 0.5, color='k', lw=0.8)
    plt.colorbar(im, ax=ax, shrink=0.8)

    fig.suptitle(f'Sensitivity Matrix U-modes — WFS vs DZ  [{label}]', fontsize=12)
    _save_and_show(fig)


def _dof_group(di):
    return 'hex' if di < 10 else ('m1m3' if di < 30 else 'm2')


def plot_vmatrices(wfs, dz, dof_idx, label):
    dof_idx = list(dof_idx)
    n = len(dof_idx)
    # y-ticks: evenly spaced DOF positions labelled by their DOF name
    tick_pos = np.unique(np.linspace(0, n - 1, min(n, 8)).astype(int))
    tick_lab = [LABELS_50DOF[dof_idx[p]] for p in tick_pos]
    # group boundaries (hexapod / M1M3 / M2 transitions)
    bounds = [p - 0.5 for p in range(1, n)
              if _dof_group(dof_idx[p]) != _dof_group(dof_idx[p - 1])]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), layout='constrained')
    for ax, (svd, name) in zip(axes, [(wfs, 'V_wfs'), (dz, 'V_dz')]):
        n_show_v = min(svd['n_keep'], svd['V'].shape[1])
        im = ax.imshow(svd['V'][:, :n_show_v], aspect='auto', cmap='RdBu_r',
                       vmin=-0.3, vmax=0.3)
        ax.set_xlabel('v-mode index')
        ax.set_ylabel('DOF')
        ax.set_title(f"{name}  ({svd['V'].shape[0]} DOFs × {n_show_v} modes)")
        ax.set_yticks(tick_pos)
        ax.set_yticklabels(tick_lab, fontsize=8)
        for b in bounds:
            ax.axhline(b, color='k', lw=0.5, alpha=0.5)
        plt.colorbar(im, ax=ax, shrink=0.8)
    fig.suptitle(f'V matrices (DOF space) — WFS vs DZ  [{label}]', fontsize=12)
    _save_and_show(fig)

In [ ]:
# Produce the SVD diagnostic plots for each (n_dof, n_keep) set
for ss in svd_sets:
    label = f"n_dof={ss['n_dof']}, n_keep={ss['n_keep']}"
    print(f"\n{'=' * 64}\n  SVD diagnostics — {label}\n{'=' * 64}")
    plot_singular_values(ss['wfs'], ss['dz'], label)
    plot_umodes(ss['wfs'], ss['dz'], label)
    plot_vmatrices(ss['wfs'], ss['dz'], ss['dof_idx'], label)

<a id='dof-recovery'></a>
## 8. DOF Recovery from WFS Mimic

For each visit, project the WFS deviation vector onto the WFS U-modes
and recover the physical DOFs.

We repeat the recovery for each `(n_dof, n_keep)` set in `dof_keep_sets`.
Each set restricts the OFC sensitivity matrix to the **leading `n_dof`**
degrees of freedom (in `LABELS_50DOF` order: M2 + Camera rigid body,
then M1M3 bending modes, then M2 bending modes) and keeps `n_keep` SVD
modes.  The per-set results are stored in `wfs_runs` and all of the
WFS-vs-FAM comparison plots in Section 9 are produced once per set.

In [ ]:
# ---- WFS deviation -> DOF recovery, reusing the per-set SVDs ----

# Flatten WFS deviation; a visit yields a solution only if its full
# deviation vector is finite.  Any NaN (a wedge with <3 donuts, or a WFS
# position off the intrinsic grid) leaves A_modes NaN, which
# recover_dof_per_visit zeros out -> an all-zero DOF vector.  Such visits
# are flagged and excluded (set to NaN) from every set's comparison.
w_wfs_flat = wfs_deviation.reshape(n_visits, -1)
wfs_finite = np.all(np.isfinite(w_wfs_flat), axis=1)


def recover_wfs_run(svd):
    """Recover per-visit DOFs for one SVD set, expanded to full 50-vector.

    Returns dof_full (n_visits, 50) with the recovered DOFs in their
    LABELS_50DOF slots and NaN elsewhere / for invalid visits.
    """
    A = np.full((n_visits, svd['n_keep']), np.nan)
    for v in range(n_visits):
        if wfs_finite[v]:
            A[v] = svd['U_eff'].T @ w_wfs_flat[v]
    dof_sub = recover_dof_per_visit(A, svd['V'], svd['Sigma'],
                                    svd['N_diag'], svd['n_keep'])
    invalid = (~wfs_finite) | np.all(dof_sub == 0, axis=1)
    dof_sub[invalid] = np.nan
    dof_full = np.full((n_visits, 50), np.nan)
    dof_full[:, svd['dof_idx']] = dof_sub
    return dof_full, invalid


wfs_runs = []
for ss in svd_sets:
    svd = ss['wfs']
    dof_full, invalid = recover_wfs_run(svd)
    wfs_runs.append(dict(n_dof=svd['n_dof'], n_keep=svd['n_keep'],
                         dof_idx=svd['dof_idx'], dof_wfs=dof_full,
                         invalid=invalid))
    n_valid = int(np.sum(np.all(np.isfinite(dof_full[:, svd['dof_idx']]), axis=1)))
    print(f"set n_dof={svd['n_dof']:>2d}, n_keep={svd['n_keep']:>2d}: "
          f"{n_valid}/{n_visits} valid visits, {int(invalid.sum())} flagged")

# Report the (set-independent) reasons visits fail — driven by non-finite
# WFS deviations.  These visits are excluded from every set.
n_bad = int((~wfs_finite).sum())
print(f"\n{n_bad}/{n_visits} visits have non-finite WFS deviation "
      f"(excluded from every set):")
if n_bad:
    print('  day_obs/seq_num : reason')
    for v in np.where(~wfs_finite)[0]:
        d = int(dz_df.iloc[v]['day_obs'])
        s = int(dz_df.iloc[v]['seq_num'])
        cnts = all_counts[v]
        bad_wedges = np.where(cnts < 3)[0]
        intr_nan = np.where(~np.all(np.isfinite(wfs_intrinsic[v]), axis=1))[0]
        reasons = []
        if bad_wedges.size:
            reasons.append(f'WFS{bad_wedges.tolist()} <3 donuts (counts={cnts.tolist()})')
        if intr_nan.size:
            reasons.append(f'intrinsic NaN at WFS{intr_nan.tolist()}')
        if not reasons:
            reasons.append('non-finite deviation')
        print(f'  {d}/{s} : ' + '; '.join(reasons))

### FAM DOFs (from build_measured_intrinsic)

Extract the FAM-derived DOFs from the dz_fits parquet for comparison.

In [ ]:
# Extract FAM DOFs from dz_fits
dof_fam = np.full((n_visits, 50), np.nan)
for di, name in enumerate(LABELS_50DOF):
    col = f'dof_{name}'
    if col in dz_df.columns:
        dof_fam[:, di] = dz_df[col].values

print(f'FAM DOF array: {dof_fam.shape}')
n_valid_fam = np.sum(np.all(np.isfinite(dof_fam), axis=1))
print(f'Valid FAM visits: {n_valid_fam}/{n_visits}')

<a id='comparison'></a>
## 9. Comparison: WFS vs FAM DOFs

The key diagnostic: how well do the WFS-derived DOFs track the
FAM-derived DOFs?  For **each `(n_dof, n_keep)` set** we show:
1. DOF time series — WFS vs FAM overlaid
2. Scatter plots — WFS DOF vs FAM DOF for each component
3. Correlation coefficients across the DOFs
4. DOF residual (WFS − FAM) mean ± std

The plotting helpers are defined first, then the full set of plots is
produced once per entry in `wfs_runs`.

In [ ]:
# ---- WFS-vs-FAM comparison plotting helpers (parametrized by DOF set) ----

hex_trans_idx = [0, 1, 2, 5, 6, 7]
hex_rot_idx = [3, 4, 8, 9]
m1m3_idx = list(range(10, 30))
m2_idx = list(range(30, 50))

DOF_GROUPS = [
    ('Hexapod Translations (μm)', hex_trans_idx),
    ('Hexapod Rotations (arcsec)', hex_rot_idx),
    ('M1M3 Bending Modes (μm)', m1m3_idx),
    ('M2 Bending Modes (μm)', m2_idx),
]
DOF_GROUPS_RESID = [
    ('Hexapod Translations', hex_trans_idx, 'μm'),
    ('Hexapod Rotations', hex_rot_idx, 'arcsec'),
    ('M1M3 Bending Modes', m1m3_idx, 'μm'),
    ('M2 Bending Modes', m2_idx, 'μm'),
]


def plot_dof_timeseries(dof_wfs, dof_fam, dof_idx, label):
    """DOF vs visit: WFS (red) and FAM (blue), grouped by mechanism."""
    dof_set = set(int(d) for d in dof_idx)
    ordinal = np.arange(dof_wfs.shape[0])
    for group_title, dof_indices in DOF_GROUPS:
        idxs = [di for di in dof_indices if di in dof_set]
        if not idxs:
            continue
        n_panels = len(idxs)
        ncols = min(5, n_panels)
        nrows = (n_panels + ncols - 1) // ncols
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(3.5 * ncols, 2.5 * nrows),
                                 layout='constrained', sharex=True)
        axes = np.atleast_1d(axes).ravel()
        for pidx, di in enumerate(idxs):
            ax = axes[pidx]
            ax.plot(ordinal, dof_fam[:, di], '.', ms=3, alpha=0.7,
                    color='tab:blue', label='FAM')
            ax.plot(ordinal, dof_wfs[:, di], '.', ms=3, alpha=0.7,
                    color='tab:red', label='WFS')
            ax.set_title(f'{LABELS_50DOF[di]}', fontsize=8)
            ax.axhline(0, color='k', lw=0.4, alpha=0.5)
            ax.grid(alpha=0.3)
            ax.tick_params(labelsize=7)
            if pidx == 0:
                ax.legend(fontsize=7)
        for k in range(n_panels, len(axes)):
            axes[k].set_visible(False)
        fig.suptitle(f'{group_title} — WFS (red) vs FAM (blue)  [{label}]',
                     fontsize=12)
        _save_and_show(fig)


def plot_dof_scatter(dof_wfs, dof_fam, dof_idx, label):
    """Per-component WFS-vs-FAM scatter + correlation; returns correlations
    aligned to dof_idx."""
    dof_idx = list(dof_idx)
    n = len(dof_idx)
    ncols = 10
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(2.2 * ncols, 2.4 * nrows),
                             layout='constrained')
    axes = np.atleast_1d(axes).ravel()
    correlations = np.full(n, np.nan)
    for pidx, di in enumerate(dof_idx):
        ax = axes[pidx]
        x = dof_fam[:, di]
        y = dof_wfs[:, di]
        good = np.isfinite(x) & np.isfinite(y)
        if good.sum() >= 5:
            ax.plot(x[good], y[good], '.', ms=2, alpha=0.5)
            corr = np.corrcoef(x[good], y[good])[0, 1]
            correlations[pidx] = corr
            lims = [min(x[good].min(), y[good].min()),
                    max(x[good].max(), y[good].max())]
            ax.plot(lims, lims, 'k-', lw=0.5, alpha=0.5)
            ax.set_title(f'{LABELS_50DOF[di]}\nr={corr:.2f}', fontsize=6)
        else:
            ax.set_title(f'{LABELS_50DOF[di]}\n(no data)', fontsize=6)
        ax.tick_params(labelsize=5)
        ax.set_aspect('equal', adjustable='datalim')
    for k in range(n, len(axes)):
        axes[k].set_visible(False)
    fig.suptitle(f'WFS DOF vs FAM DOF — scatter + correlation  [{label}]',
                 fontsize=12)
    fig.supxlabel('FAM DOF', fontsize=10)
    fig.supylabel('WFS DOF', fontsize=10)
    _save_and_show(fig)
    return correlations


def plot_corr_bar(correlations, dof_idx, label):
    """Bar chart of per-DOF Pearson r (correlations aligned to dof_idx)."""
    dof_idx = list(dof_idx)
    n = len(dof_idx)
    fig, ax = plt.subplots(figsize=(max(8, 0.32 * n), 5), layout='constrained')
    x_pos = np.arange(n)
    colors_bar = np.where(correlations > 0.7, 'tab:green',
                  np.where(correlations > 0.3, 'tab:orange', 'tab:red'))
    ax.bar(x_pos, correlations, color=colors_bar, alpha=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([LABELS_50DOF[di] for di in dof_idx],
                       rotation=90, fontsize=7)
    ax.set_ylabel('Pearson r (WFS vs FAM)')
    ax.set_title(f'DOF Recovery Correlation: WFS Mimic vs FAM  [{label}]')
    ax.axhline(0.7, color='green', ls='--', lw=0.8, alpha=0.5, label='r=0.7')
    ax.axhline(0.3, color='orange', ls='--', lw=0.8, alpha=0.5, label='r=0.3')
    ax.axhline(0, color='k', lw=0.5)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    n_good = int(np.sum(correlations > 0.7))
    n_fair = int(np.sum((correlations > 0.3) & (correlations <= 0.7)))
    n_poor = int(np.sum(correlations <= 0.3))
    ax.text(0.98, 0.95,
            f'r>0.7: {n_good}  |  0.3<r≤0.7: {n_fair}  |  r≤0.3: {n_poor}',
            transform=ax.transAxes, ha='right', va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    _save_and_show(fig)
    print(f'  [{label}] {n_good}/{n} DOFs well-recovered (r>0.7), '
          f'{n_fair} fair, {n_poor} poor')


def plot_dof_residual(dof_wfs, dof_fam, dof_idx, label):
    """Mean and std of the (WFS - FAM) residual, grouped by mechanism."""
    dof_set = set(int(d) for d in dof_idx)
    resid = dof_wfs - dof_fam
    groups = [(t, [di for di in idx if di in dof_set], u)
              for t, idx, u in DOF_GROUPS_RESID]
    groups = [g for g in groups if g[1]]
    nrows = len(groups)
    fig, axes = plt.subplots(nrows, 1, figsize=(15, 3.2 * nrows),
                             layout='constrained')
    axes = np.atleast_1d(axes).ravel()
    for ax, (title, idx_list, unit) in zip(axes, groups):
        x = np.arange(len(idx_list))
        means = np.array([np.nanmean(resid[:, di]) for di in idx_list])
        stds = np.array([np.nanstd(resid[:, di]) for di in idx_list])
        ax.bar(x - 0.2, means, width=0.4, color='tab:blue',
               alpha=0.8, label='mean')
        ax.bar(x + 0.2, stds, width=0.4, color='tab:orange',
               alpha=0.8, label='std')
        ax.set_xticks(x)
        ax.set_xticklabels([LABELS_50DOF[di] for di in idx_list],
                           rotation=45, ha='right', fontsize=8)
        ax.set_ylabel(f'Residual ({unit})')
        ax.set_title(f'{title} — WFS − FAM residual')
        ax.axhline(0, color='k', lw=0.5)
        ax.grid(axis='y', alpha=0.3)
        ax.legend(fontsize=8)
    fig.suptitle(f'DOF Residual (WFS Mimic − FAM): Mean ± Std  [{label}]',
                 fontsize=12)
    _save_and_show(fig)

In [ ]:
# Produce the full set of WFS-vs-FAM comparison plots for each set
for run in wfs_runs:
    nd, nk = run['n_dof'], run['n_keep']
    label = f'n_dof={nd}, n_keep={nk}'
    dof_wfs = run['dof_wfs']
    dof_idx = run['dof_idx']

    print(f"\n{'=' * 64}\n  {label}\n{'=' * 64}")

    plot_dof_timeseries(dof_wfs, dof_fam, dof_idx, label)
    correlations = plot_dof_scatter(dof_wfs, dof_fam, dof_idx, label)
    plot_corr_bar(correlations, dof_idx, label)
    plot_dof_residual(dof_wfs, dof_fam, dof_idx, label)

<a id='fwhm'></a>
## 10. PSF FWHM-Equivalent: Before vs After DOF Adjustment

The wavefront deviation at each donut is converted to an equivalent PSF
FWHM contribution (arcsec) using the WEP convention
`convertZernikesToPsfWidth` — the per-Noll dPSF/dZ gradient factors —
quadrature-summed over the pupil Zernikes.

For every donut over the field of view we compute the FWHM-equivalent of:

1. **Before adjustment** — the raw deviation `measured − intrinsic`
   (Z4 height-corrected, as in the WFS path).
2. **After FAM adjustment** — deviation minus the wavefront predicted by
   the **FAM** DOFs at that donut's field position
   (`S(thx, thy) @ dof_FAM`).
3. **After WFS-mimic adjustment** — same, using the **WFS-mimic** DOFs
   recovered for each `(n_dof, n_keep)` set.

The DOF-predicted wavefront uses the OFC sensitivity matrix at each
donut's field position.  Because `dz_sens.evaluate` is slow per point,
the sensitivity matrix is evaluated once on a coarse focal-plane grid
(`fwhm_field_grid_n` points across the field diameter) and bilinearly
interpolated to each donut.  We then show, per set, a violin plot of the
per-donut FWHM-equivalent vs image ordinal for the three cases, plus a
summary violin pooling all donuts across all visits to compare every
case at once.

In [ ]:
# ---- Per-donut PSF FWHM-equivalent: before / after FAM / after WFS ----
from lsst.ts.wep.utils import convertZernikesToPsfWidth
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kw):
        return x

# Per-Noll conversion factors (arcsec FWHM per μm of wavefront), aligned
# to our pupil Zernikes iZs.  convertZernikesToPsfWidth assumes a
# contiguous array starting at Noll 4, so probe it with a unit vector.
_L_full = int(iZs_arr.max()) - 4 + 1
psf_conv_full = np.asarray(convertZernikesToPsfWidth(np.ones(_L_full)))
conv_iZs = psf_conv_full[iZs_arr - 4]          # (n_j,) arcsec/μm per Zernike


def fwhm_equiv(dev):
    """Quadrature PSF-FWHM (arcsec) from wavefront dev (..., n_j) in μm."""
    dfwhm = np.asarray(dev) * conv_iZs
    return np.sqrt(np.sum(dfwhm**2, axis=-1))


# Per-donut measured Zernikes (Z4 height-corrected) and deviation
zk_meas_all = np.stack(donut_df[f'zk_{coord_sys}'].values).astype(float)
if Z4hgt is not None and 4 in iZidx:
    zk_meas_all[:, iZidx[4]] = zk_meas_all[:, iZidx[4]] - Z4hgt

thx_all_deg = np.degrees(np.asarray(donut_df[f'thx_{coord_sys}'], dtype=float))
thy_all_deg = np.degrees(np.asarray(donut_df[f'thy_{coord_sys}'], dtype=float))
zk_intr_all = intrinsic_grid.evaluate(thx_all_deg, thy_all_deg)   # (N, n_j)
dev_before_all = zk_meas_all - zk_intr_all                        # (N, n_j)

# Build a bilinear interpolator of the (selected-Zernike, DOF) sensitivity
# matrix over a coarse focal-plane grid.  dz_sens.evaluate is called ONCE
# on the grid instead of per donut.
_R = max(fp_radius_basis,
         float(np.nanmax(np.abs(thx_all_deg))),
         float(np.nanmax(np.abs(thy_all_deg)))) * 1.001
_gx = np.linspace(-_R, _R, fwhm_field_grid_n)
_gy = np.linspace(-_R, _R, fwhm_field_grid_n)
_GX, _GY = np.meshgrid(_gx, _gy, indexing='ij')
_grid_pts = [[float(x), float(y)] for x, y in zip(_GX.ravel(), _GY.ravel())]
print(f'Evaluating sensitivity matrix on {fwhm_field_grid_n}x{fwhm_field_grid_n} '
      f'field grid (±{_R:.3f} deg, {_R * 2 / (fwhm_field_grid_n - 1):.3f} deg spacing)...')
_S_grid_flat = dz_sens.evaluate(_grid_pts, 0.0)            # (G, n_zk_full, n_dof)
_S_grid = _S_grid_flat[:, zn_idx, :].reshape(
    fwhm_field_grid_n, fwhm_field_grid_n, n_j, _S_grid_flat.shape[2])
S_field_interp = RegularGridInterpolator(
    (_gx, _gy), _S_grid, method='linear', bounds_error=False, fill_value=None)


def sensitivity_at(thx_deg, thy_deg):
    """Bilinearly interpolated sensitivity (n_pts, n_j, n_dof)."""
    return S_field_interp(np.column_stack([thx_deg, thy_deg]))


# Per-visit, per-donut FWHM-equivalent for each case.
fwhm_before_by_visit = [None] * n_visits
fwhm_after_fam_by_visit = [None] * n_visits
fwhm_after_wfs_by_visit = [[None] * n_visits for _ in wfs_runs]

for v in tqdm(range(n_visits), desc='FWHM per visit'):
    d = int(dz_df.iloc[v]['day_obs'])
    s = int(dz_df.iloc[v]['seq_num'])
    rows = np.where((dobs_arr == d) & (snum_arr == s))[0]
    if rows.size == 0:
        continue

    dev_b = dev_before_all[rows]                  # (n_d, n_j)
    fwhm_before_by_visit[v] = fwhm_equiv(dev_b)

    # Sensitivity matrix interpolated at each donut field position
    S_sel = sensitivity_at(thx_all_deg[rows], thy_all_deg[rows])  # (n_d, n_j, 50)

    # After FAM adjustment
    if np.all(np.isfinite(dof_fam[v])):
        wf_fam = np.einsum('djf,f->dj', S_sel, dof_fam[v])
        fwhm_after_fam_by_visit[v] = fwhm_equiv(dev_b - wf_fam)

    # After WFS-mimic adjustment, per set
    for ri, run in enumerate(wfs_runs):
        di = run['dof_idx']
        dof_v = run['dof_wfs'][v]
        if np.all(np.isfinite(dof_v[di])):
            wf_wfs = np.einsum('djf,f->dj', S_sel[:, :, di], dof_v[di])
            fwhm_after_wfs_by_visit[ri][v] = fwhm_equiv(dev_b - wf_wfs)


def _clean_fwhm(arr, min_n=3):
    """Finite values from a per-visit FWHM array, or None if too few."""
    if arr is None:
        return None
    a = np.asarray(arr, dtype=float)
    a = a[np.isfinite(a)]
    return a if a.size >= min_n else None


# Quick numeric summary (median over all donuts, all visits)
def _pool(per_visit):
    arrs = [_clean_fwhm(a) for a in per_visit]
    arrs = [a for a in arrs if a is not None]
    return np.concatenate(arrs) if arrs else np.array([])


_before_pool = _pool(fwhm_before_by_visit)
_fam_pool = _pool(fwhm_after_fam_by_visit)
print('\nMedian PSF FWHM-equivalent over all donuts [arcsec]:')
print(f'  before adjustment : {np.median(_before_pool):.4f}'
      if _before_pool.size else '  before adjustment : (no data)')
print(f'  after FAM         : {np.median(_fam_pool):.4f}'
      if _fam_pool.size else '  after FAM         : (no data)')
for ri, run in enumerate(wfs_runs):
    pool = _pool(fwhm_after_wfs_by_visit[ri])
    tag = f"after WFS n_dof={run['n_dof']}, n_keep={run['n_keep']}"
    print(f'  {tag:30s}: {np.median(pool):.4f}'
          if pool.size else f'  {tag:30s}: (no data)')

In [ ]:
# ---- Per-set violin plots: FWHM-equiv vs image ordinal ----
def _add_violins(ax, per_visit, x_base, offset, color, label, width=0.22):
    data, pos = [], []
    for v, arr in enumerate(per_visit):
        a = _clean_fwhm(arr)
        if a is not None:
            data.append(a)
            pos.append(x_base[v] + offset)
    if not data:
        return
    parts = ax.violinplot(data, positions=pos, widths=width,
                          showmedians=True, showextrema=False)
    for body in parts['bodies']:
        body.set_facecolor(color)
        body.set_edgecolor(color)
        body.set_alpha(0.5)
    if 'cmedians' in parts:
        parts['cmedians'].set_color(color)
    ax.plot([], [], color=color, lw=6, alpha=0.5, label=label)  # legend proxy


x_base = np.arange(n_visits)
tick_step = max(1, n_visits // 40)

for ri, run in enumerate(wfs_runs):
    label = f"n_dof={run['n_dof']}, n_keep={run['n_keep']}"
    fig, ax = plt.subplots(figsize=(max(12, 0.45 * n_visits), 6),
                           layout='constrained')
    _add_violins(ax, fwhm_before_by_visit, x_base, -0.26, 'tab:gray',
                 'before adjustment')
    _add_violins(ax, fwhm_after_fam_by_visit, x_base, 0.0, 'tab:blue',
                 'after FAM')
    _add_violins(ax, fwhm_after_wfs_by_visit[ri], x_base, 0.26, 'tab:red',
                 f'after WFS ({label})')
    ax.set_xticks(x_base[::tick_step])
    ax.set_xticklabels(x_base[::tick_step], fontsize=7)
    ax.set_xlabel('image ordinal')
    ax.set_ylabel('PSF FWHM-equivalent [arcsec]')
    ax.set_title(f'FWHM-equivalent before vs after DOF adjustment  [{label}]')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=9, loc='upper right')
    _save_and_show(fig)

In [ ]:
# ---- Summary violin: all cases pooled over every donut / visit ----
cases = [('before', _pool(fwhm_before_by_visit), 'tab:gray'),
         ('after FAM', _pool(fwhm_after_fam_by_visit), 'tab:blue')]
wfs_colors = ['tab:red', 'tab:orange', 'tab:green', 'tab:purple']
for ri, run in enumerate(wfs_runs):
    cases.append((f"after WFS\nn_dof={run['n_dof']}, n_keep={run['n_keep']}",
                  _pool(fwhm_after_wfs_by_visit[ri]),
                  wfs_colors[ri % len(wfs_colors)]))

cases = [(lab, arr, col) for (lab, arr, col) in cases if arr.size]
data = [arr for _, arr, _ in cases]
labels = [lab for lab, _, _ in cases]
colors = [col for _, _, col in cases]
pos = np.arange(len(data))

fig, ax = plt.subplots(figsize=(2.0 + 1.6 * len(data), 6), layout='constrained')
parts = ax.violinplot(data, positions=pos, widths=0.8, showmedians=True,
                      showextrema=True)
for body, col in zip(parts['bodies'], colors):
    body.set_facecolor(col)
    body.set_edgecolor(col)
    body.set_alpha(0.5)
for key in ('cbars', 'cmins', 'cmaxes', 'cmedians'):
    if key in parts:
        parts[key].set_color('0.3')
        parts[key].set_linewidth(1.0)

for i, arr in enumerate(data):
    med = np.median(arr)
    ax.text(i, med, f'  {med:.3f}"', va='center', ha='left', fontsize=9,
            fontweight='bold')

ax.set_xticks(pos)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('PSF FWHM-equivalent [arcsec]')
ax.set_title('PSF FWHM-equivalent — all donuts, all visits '
             '(median annotated)')
ax.grid(axis='y', alpha=0.3)
_save_and_show(fig)

print('Summary (pooled over all donuts/visits) [arcsec]:')
print(f'{"case":40s} {"median":>8s} {"mean":>8s} {"p90":>8s} {"n_donuts":>10s}')
for lab, arr, _ in cases:
    flat_lab = lab.replace('\n', ' ')
    print(f'{flat_lab:40s} {np.median(arr):8.4f} {np.mean(arr):8.4f} '
          f'{np.percentile(arr, 90):8.4f} {arr.size:10d}')

In [ ]:
# ---- Finalize: close the collected PDF ----
if _pdf is not None:
    _pdf.close()
    print(f'Saved all plots to {pdf_path}')